# Passport Photo Validation - Check Visualizer

This notebook is for running individual validation checks and visualizing their internal results for debugging and development.

In [4]:
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# TODO: use this to create html/md reports later, i.e. with images embedded but no code necessarily shown

# Add the src directory to the Python path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from edutap.image_api.core.validator import PassportPhotoValidator
from edutap.image_api.core.models import ValidationContext
from edutap.image_api.core.checks import background, lighting, geometry, features, quality

## Setup and Image Loading

In [5]:
# Initialize the validator (we need it for its helper methods)
validator = PassportPhotoValidator()

# Select an image to analyze
image_path = Path.cwd().parent / 'tests' / 'test-images' / '6.png'
image_np = cv2.imread(str(image_path))

if image_np is None:
    print(f"Error: Could not load image from {image_path}")
else:
    print(f"Image loaded: {image_np.shape[1]}x{image_np.shape[0]} pixels")
    plt.figure(figsize=(6, 8))
    plt.imshow(cv2.cvtColor(image_np, cv2.COLOR_BGR2RGB))
    plt.title('Original Image')
    plt.axis('off')
    plt.show()

Error: Could not load image from /home/simon/workspaces/lmu/edutap.image_api/tests/test-images/6.png


I0000 00:00:1763678927.850812  160555 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763678927.917198  161245 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.274.02), renderer: NVIDIA GeForce RTX 3090/PCIe/SSE2
W0000 00:00:1763678927.921043  161240 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1763678927.922381  160555 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763678927.936153  161262 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.274.02), renderer: NVIDIA GeForce RTX 3090/PCIe/SSE2
I0000 00:00:1763678927.937774  160555 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
W0000 00:00:1763678927.940873  161255 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000

## Create a Validation Context

To test a check function, we first need to create a `ValidationContext` and populate it with the necessary data (like face detection results) that the check function expects.

In [6]:
context = ValidationContext(image_np=image_np)

# Run face detection and populate the context
detection_results = validator._detect_faces(context)

if not detection_results or not detection_results.detections:
    print("Could not detect a face in the image.")
else:
    # Use the first detected face
    validator._populate_face_data(context, detection_results.detections[0])
    # Also populate the segmentation mask for checks that need it
    validator._populate_segmentation_data(context)
    print("ValidationContext created and populated with face and segmentation data.")

error: OpenCV(4.11.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


## Visualize `check_background`

In [4]:
# Run the background check with debug=True to get visualization data
# Run individual background checks with debug=True to get visualization data
brightness_data = background.check_background_brightness(context, debug=True)
texture_data = background.check_background_texture(context, debug=True)
uniformity_data = background.check_background_uniformity(context, debug=True)
consistency_data = background.check_background_consistency(context, debug=True)
clutter_data = background.check_background_clutter(context, debug=True)
# Combine the data for visualization
background_viz_data = {}
if texture_data:
    background_viz_data.update(texture_data)
if clutter_data:
    background_viz_data.update(clutter_data)
if uniformity_data:
    background_viz_data.update(uniformity_data)
if consistency_data:
    background_viz_data.update(consistency_data)
if brightness_data:
    background_viz_data.update(brightness_data)
# Add mask data for visualization
if context.segmentation_mask is not None:
    background_mask_binary = (context.segmentation_mask < 0.5).astype(np.uint8)
    background_viz_data['background_mask_float'] = context.segmentation_mask
    background_viz_data['background_mask'] = background_mask_binary * 255


if background_viz_data and 'background_mask' in background_viz_data:
    print("Successfully received visualization data from check_background.")

    # The check returns comprehensive analysis data
    float_mask = background_viz_data['background_mask_float']
    binary_mask = background_viz_data['background_mask']
    edge_map = background_viz_data.get('edge_density_map')
    rect_viz = background_viz_data.get('rectangles_visualization')

    # Get metrics
    color_coh = background_viz_data.get('color_coherence', 0)
    gradient_coh = background_viz_data.get('gradient_coherence', 0)
    edge_dens = background_viz_data.get('edge_density', 0)
    rect_count = background_viz_data.get('rectangle_count', 0)

    # Create visualizations
    overlay = cv2.cvtColor(image_np, cv2.COLOR_BGR2RGB)

    # Create figure with 6 subplots in 2 rows
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[1, 0])
    ax5 = fig.add_subplot(gs[1, 1])
    ax6 = fig.add_subplot(gs[1, 2])

    # 1. Segmentation mask
    ax1.imshow(float_mask, cmap='viridis')
    ax1.set_title('Segmentation Float Mask')
    ax1.axis('off')

    # 2. Background binary mask
    ax2.imshow(binary_mask, cmap='gray')
    ax2.set_title('Background Binary Mask')
    ax2.axis('off')

    # 3. Edge density map
    if edge_map is not None:
        ax3.imshow(edge_map, cmap='hot')
        ax3.set_title(f'Edge Density Map ({edge_dens:.2%})')
        ax3.axis('off')

    # 4. Sampled rectangles (KEY VISUALIZATION)
    if rect_viz is not None:
        ax4.imshow(cv2.cvtColor(rect_viz, cv2.COLOR_BGR2RGB))
        ax4.set_title(f'Sampled Rectangles ({rect_count} regions)')
        ax4.axis('off')

    # 5. Original image
    ax5.imshow(overlay)
    ax5.set_title('Original Image')
    ax5.axis('off')

    # 6. Metrics summary
    ax6.axis('off')

    # Determine pass/fail for each metric
    color_coh_pass = color_coh <= 0.15
    gradient_coh_pass = gradient_coh <= 3.0 or color_coh > 0.15  # Skip if color failed
    edge_dens_pass = edge_dens < 0.15

    metrics_text = f"""Background Analysis Metrics:

Rectangle Count: {rect_count}
(Need ≥3 clean regions)

Color Coherence: {color_coh:.3f} {'✓' if color_coh_pass else '✗'}
(< 0.15 = smooth gradient) [PRIMARY]

Gradient Coherence: {gradient_coh:.3f} {'✓' if gradient_coh_pass else '✗'}
(< 3.0 = smooth) [SECONDARY]

Edge Density: {edge_dens:.2%} {'✓' if edge_dens_pass else '✗'}
(< 15% = plain)

Overall: {'✓ PASS' if color_coh_pass and edge_dens_pass else '✗ FAIL'}
"""
    ax6.text(0.1, 0.5, metrics_text, transform=ax6.transAxes,
             fontsize=11, verticalalignment='center', family='monospace',
             bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

    plt.suptitle('`check_background` Visualization (Hierarchical Analysis)', fontsize=16, weight='bold')
    plt.show()

    print("\nMetrics:")
    print(f"  Rectangle Count: {rect_count} (need ≥3)")
    print(f"  Color Coherence: {color_coh:.3f} (< 0.15 = smooth) [PRIMARY CHECK]")
    print(f"  Gradient Coherence: {gradient_coh:.3f} (< 3.0 = smooth) [SECONDARY]")
    print(f"  Edge Density: {edge_dens:.2%} (< 15% = plain)")

    print("\nIssues found by this check:")
    # Clear previous issues from context for this isolated check
    context.issues = []
    background.check_background_brightness(context, debug=False)
    background.check_background_texture(context, debug=False)
    background.check_background_uniformity(context, debug=False)
    background.check_background_consistency(context, debug=False)
    background.check_background_clutter(context, debug=False) # Rerun without debug to populate issues
    if context.issues:
        for issue in context.issues:
            print(f"- {issue.message}")
    else:
        print("✓ No issues found - background passes all checks!")
else:
    print("Did not receive visualization data.")





AttributeError: module 'edutap.image_api.core.checks.background' has no attribute 'check_background'

## Visualize `check_distortions` (BRISQUE)

This check uses the BRISQUE model to provide quality scores for both the overall image and the face region.

In [ ]:
# Run the distortions check on the face with debug=True
error_thresh = 40.0
warning_thresh = 30.0

# Clear context issues before this specific check
context.issues = []
face_distortions_viz_data = quality.check_distortions(context,
                                                      area='face',
                                                      error_threshold=error_thresh,
                                                      warning_threshold=warning_thresh,
                                                      debug=True)

if face_distortions_viz_data:
    print(f"Successfully received visualization data from check_distortions (face).")
    score = face_distortions_viz_data['brisque_score']
    analyzed_image = face_distortions_viz_data['analyzed_image']

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))

    # Display the image that was analyzed
    ax.imshow(analyzed_image, cmap='gray')
    ax.set_title('Face Region Analyzed by BRISQUE')
    ax.axis('off')

    # Add text with the score
    status = "Poor Quality (Error)" if score > error_thresh else "OK Quality (Warning)" if score > warning_thresh else "Good Quality"
    color = "red" if "Error" in status else "orange" if "Warning" in status else "green"

    fig.suptitle(f"Face BRISQUE Score: {score:.2f} -> {status}", fontsize=16, weight='bold', color=color)
    plt.show()

    # Print issues found by this check
    print("\nIssues found by this check:")
    if context.issues:
        for issue in context.issues:
            print(f"- {issue.message}")
    else:
        print("No issues found.")
else:
    print("Did not receive visualization data from face distortions check.")

## Visualize `check_distortions` on Overall Image

This check analyzes the entire image for general quality issues.

In [ ]:
# Run the distortions check on the full image with debug=True
context.issues = []
overall_distortions_viz_data = quality.check_distortions(context,
                                                         area='full',
                                                         error_threshold=error_thresh,
                                                         warning_threshold=warning_thresh,
                                                         debug=True)

if overall_distortions_viz_data:
    print(f"Successfully received visualization data from check_distortions (full).")
    score = overall_distortions_viz_data['brisque_score']
    analyzed_image = overall_distortions_viz_data['analyzed_image']

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(analyzed_image, cmap='gray')
    ax.set_title('Overall Image Analyzed by BRISQUE')
    ax.axis('off')

    status = "Poor Quality (Error)" if score > error_thresh else "OK Quality (Warning)" if score > warning_thresh else "Good Quality"
    color = "red" if "Error" in status else "orange" if "Warning" in status else "green"
    fig.suptitle(f"Overall BRISQUE Score: {score:.2f} -> {status}", fontsize=16, weight='bold', color=color)
    plt.show()

    print("\nIssues found by this check:")
    if context.issues:
        for issue in context.issues:
            print(f"- {issue.message}")
    else:
        print("No issues found.")
else:
    print("Did not receive visualization data from overall distortions check.")
